# Day 2 Tutorial：回归评价指标

## Goal

从四个预测建立逐样本误差表，手算并核对 MAE、RMSE、R²；教学输出不是个人实验成果。

## Setup

使用固定人工数组，不加载 ESOL，不访问 test。

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_true = np.array([-3.0, -2.0, -1.0, 0.0])
y_pred = np.array([-2.5, -2.4, -0.2, -0.1])

print('y_true shape:', y_true.shape)
print('y_pred shape:', y_pred.shape)

y_true shape: (4,)
y_pred shape: (4,)


## Steps

先保留每个样本的误差，再汇总指标。

In [2]:
error_table = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred})
error_table['error'] = error_table['y_pred'] - error_table['y_true']
error_table['absolute_error'] = error_table['error'].abs()
error_table['squared_error'] = error_table['error'] ** 2
error_table

,y_true,y_pred,error,absolute_error,squared_error
0,-3.0,-2.5,0.5,0.5,0.25
1,-2.0,-2.4,-0.4,0.4,0.16
2,-1.0,-0.2,0.8,0.8,0.64
3,0.0,-0.1,-0.1,0.1,0.01


In [3]:
def regression_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float).reshape(-1)
    predicted = np.asarray(predicted, dtype=float).reshape(-1)
    if actual.shape != predicted.shape:
        raise ValueError('actual 与 predicted 必须有相同 shape')
    if actual.size < 2:
        raise ValueError('至少需要两个样本')
    if not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('输入不能包含 NaN 或无穷')
    return {
        'mae': float(mean_absolute_error(actual, predicted)),
        'rmse': float(np.sqrt(mean_squared_error(actual, predicted))),
        'r2': float(r2_score(actual, predicted)),
    }

scores = regression_metrics(y_true, y_pred)
scores

{'mae': 0.45, 'rmse': 0.51478150704935, 'r2': 0.788}

## Checks

用独立公式核对三项，并确认负 R² 是合法结果。

In [4]:
manual_mae = error_table['absolute_error'].mean()
manual_rmse = np.sqrt(error_table['squared_error'].mean())
manual_r2 = 1 - (
    error_table['squared_error'].sum()
    / ((y_true - y_true.mean()) ** 2).sum()
)

assert np.isclose(manual_mae, scores['mae'])
assert np.isclose(manual_rmse, scores['rmse'])
assert np.isclose(manual_r2, scores['r2'])

bad_scores = regression_metrics(y_true, np.array([3.0, 2.0, 1.0, 0.0]))
assert bad_scores['r2'] < 0
print('manual checks passed')
print('negative R² example:', round(bad_scores['r2'], 3))

manual checks passed
negative R² example: -10.2


## Next Steps

关闭本教程，独立完成 `03_exercises.md`；学习当天再把教程复制到 `experiments/day02_metrics/`，从空内核运行并写自己的解释。